# Curriculum catalogue selection

Create stage-specific target catalogues from the cleaned merged catalogue.

The radius thresholds are learned once from catalogue A. Those fixed thresholds are then applied to the cleaned merged ABC catalogue. For early curriculum stages, smaller bubbles are intentionally omitted from the stage catalogue, so they become background when `galaxyMaskGeneration.ipynb` creates the binary mask.


In [60]:
import os

import pandas as pd


In [61]:
CATALOGUE_A_PATH = "../data/raw/catalogue/jwst_bubble_properties_A.txt"
INPUT_PATH = "../data/processed/metadata/catalogueMerged.txt"
OUTPUT_DIR = "../data/processed/metadata/curriculum"

SIZE_CLASSES = ["tiny", "small", "medium", "large", "giants"]
EDGE_CLASS = "edgeLarge"
GROUP_ORDER = SIZE_CLASSES + [EDGE_CLASS]

SIZE_PERCENTILES = [0.20, 0.40, 0.60, 0.80]
EDGE_PERCENTILE = 0.995

CURRICULUM_STAGES = {
    "stage01": ["giants", "edgeLarge"],
    # "stage02_large_plus": ["large", "giants", "edgeLarge"],
    # "stage03_medium_plus": ["medium", "large", "giants", "edgeLarge"],
    # "stage04_small_plus": ["small", "medium", "large", "giants", "edgeLarge"],
    # "stage05_all": ["tiny", "small", "medium", "large", "giants", "edgeLarge"],
}

ACTIVE_STAGE = "stage01"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [62]:
# CATALOGUE_A_REQUIRED_COLUMNS = {
#     "ID",
#     "AVG_RAD_PC",
# }

# MERGED_REQUIRED_COLUMNS = {
#     "GLOBAL_ID",
#     "ID",
#     "RA_DMS",
#     "DEC_DMS",
#     "SEMI_MAJ_PC",
#     "SEMI_MIN_PC",
#     "AVG_RAD_PC",
#     "PA_DEG",
#     "CATALOGUE",
# }


In [63]:
# def validate_columns(df, required_columns, input_path):
#     missing_columns = sorted(required_columns - set(df.columns))

#     if missing_columns:
#         raise ValueError(
#             f"{input_path} is missing required columns: {missing_columns}"
#         )


def build_catalogue_a_radius_bins(catalogue_a):
    radius = catalogue_a["AVG_RAD_PC"]

    percentile_values = radius.quantile(SIZE_PERCENTILES + [EDGE_PERCENTILE])
    q20, q40, q60, q80 = [percentile_values[p] for p in SIZE_PERCENTILES]
    edge_threshold = percentile_values[EDGE_PERCENTILE]

    finite_edges = [q20, q40, q60, q80, edge_threshold]
    if len(set(finite_edges)) != len(finite_edges):
        raise ValueError(
            "Catalogue A radius percentile edges are not unique. "
            "Adjust the binning rule before assigning size classes."
        )

    bin_edges = [float("-inf"), q20, q40, q60, q80, edge_threshold, float("inf")]

    radius_bounds = pd.DataFrame(
        {
            "SIZE_CLASS": GROUP_ORDER,
            "lower_bound_pc": bin_edges[:-1],
            "upper_bound_pc": bin_edges[1:],
        }
    )

    return bin_edges, radius_bounds


def assign_size_classes(df, bin_edges):
    df = df.copy()

    df["SIZE_CLASS"] = pd.cut(
        df["AVG_RAD_PC"],
        bins=bin_edges,
        labels=GROUP_ORDER,
        right=False,
    ).astype(str)

    if df["SIZE_CLASS"].isna().any():
        raise ValueError("Some bubbles could not be assigned to a size class.")

    return df

In [64]:
catalogue_a = pd.read_csv(CATALOGUE_A_PATH)
# validate_columns(catalogue_a, CATALOGUE_A_REQUIRED_COLUMNS, CATALOGUE_A_PATH)

bin_edges, radius_bounds = build_catalogue_a_radius_bins(catalogue_a)
catalogue_a_labelled = assign_size_classes(catalogue_a, bin_edges)

radius_bounds = radius_bounds.merge(
    catalogue_a_class_summary,
    on="SIZE_CLASS",
    how="left",
)

print("Catalogue A radius bounds used for all curriculum stages:")
print(radius_bounds.to_string(index=False))


Catalogue A radius bounds used for all curriculum stages:
SIZE_CLASS  lower_bound_pc  upper_bound_pc  catalogue_a_count  catalogue_a_min_radius_pc  catalogue_a_max_radius_pc
      tiny            -inf           18.00                315                          6                         17
     small           18.00           24.00                348                         18                         23
    medium           24.00           30.00                331                         24                         29
     large           30.00           41.00                351                         30                         40
    giants           41.00          228.07                340                         41                        227
 edgeLarge          228.07             inf                  9                        229                        552


In [65]:
df = pd.read_csv(INPUT_PATH)
# validate_columns(df, MERGED_REQUIRED_COLUMNS, INPUT_PATH)

df_labelled = assign_size_classes(df, bin_edges)
df_labelled = df_labelled.sort_values("GLOBAL_ID").reset_index(drop=True)

for stage_name, target_classes in CURRICULUM_STAGES.items():
    df_labelled[f"IN_{stage_name}"] = df_labelled["SIZE_CLASS"].isin(target_classes)

print("Loaded cleaned merged catalogue:", len(df_labelled), "bubbles")
print("\nSize class counts by catalogue using catalogue A bounds:")
print(
    pd.crosstab(
        df_labelled["CATALOGUE"],
        df_labelled["SIZE_CLASS"],
    )
    .reindex(columns=GROUP_ORDER, fill_value=0)
    .to_string()
)


Loaded cleaned merged catalogue: 3259 bubbles

Size class counts by catalogue using catalogue A bounds:
SIZE_CLASS  tiny  small  medium  large  giants  edgeLarge
CATALOGUE                                                
A            315    348     331    341     298          6
B             18    102     157    247     307          6
C             45    142     179    191     221          5


In [66]:
base_output_columns = list(df.columns) + ["SIZE_CLASS"]

bounds_path = os.path.join(OUTPUT_DIR, "catalogueA_sizeClassBounds.csv")

# df_labelled.to_csv(stratified_path, index=False)
radius_bounds.to_csv(bounds_path, index=False)
print("Saved catalogue A size-class bounds:", bounds_path)


Saved catalogue A size-class bounds: ../data/processed/metadata/curriculum/catalogueA_sizeClassBounds.csv


In [67]:
# if ACTIVE_STAGE not in CURRICULUM_STAGES:
#     raise ValueError(
#         f"ACTIVE_STAGE must be one of: {list(CURRICULUM_STAGES)}"
#     )

active_catalogue_path = os.path.join(OUTPUT_DIR, f"{ACTIVE_STAGE}.txt")
# active_df = pd.read_csv(active_catalogue_path)

print("Active stage:", ACTIVE_STAGE)
print("Active target classes:", CURRICULUM_STAGES[ACTIVE_STAGE])
print("Active stage bubbles:", len(active_df))
print("\nUse this path in galaxyMaskGeneration.ipynb:")


Active stage: stage01
Active target classes: ['giants', 'edgeLarge']
Active stage bubbles: 843

Use this path in galaxyMaskGeneration.ipynb:
